# Globally Optimal Robust Plane Fitting

_More coming soon!_

I developed this notebook in tandem with a technical report available on GitHub at [josephrhobbs/robust-plane-fitting](https://github.com/josephrhobbs/robust-plane-fitting).

__License__: MIT

## Imports

In [ ]:
from abc import ABC, abstractmethod
import cvxpy as cp
from matplotlib import pyplot as plt
import numpy as np
import scipy as sp
from time import perf_counter
from tqdm import tqdm

## Utilities

## Compute Plane Fit Error

Note that this is the error between the _predicted plane_ and the _ground-truth plane_.  The error is represented as a geodesic distance on the manifold of possible planes.

In [ ]:
def geodesic(
    normal1: np.array,
    normal2: np.array,
) -> float:
    """
    Compute the geodesic distance between two representations
        of a plane.
    """
    denom = np.linalg.norm(normal1) * np.linalg.norm(normal2)
    d1 = np.power(np.acos(
        np.clip(np.abs(normal1 @ normal2 / denom), -1, 1)
    ), 2)
    return np.sqrt(d1)

### Plot 3D Data

In [ ]:
def plot_3d_data(
    point_clouds: list[np.ndarray],
    labels=None,
    title="Point Clouds",
):
    """
    Given a collection of point clouds, plot them.
    """
    # Construct plot
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    # Process labels
    if labels is None:
        labels = [None] * len(point_clouds)

    # Plot each point cloud
    for pc, label in zip(point_clouds, labels):
        ax.scatter(pc[0, :], pc[1, :], pc[2, :], label=label)

    if any(l is not None for l in labels):
        plt.legend()
    plt.suptitle(title)
    plt.show()

## Robust Plane Fitting (RPF)

### Define Solver Base Class

In [ ]:
class Solver(ABC):
    """
    Abstract base class for RPF solver.
    """
    def __init__(self):
        """
        Initialize this solver.
        """
        pass

    @abstractmethod
    def solve(
        self,
        x: np.ndarray,
    ) -> tuple[float, np.ndarray, float]:
        pass

### Define RPF Solver

In [ ]:
class EigenvalueRPFSolver(Solver):
    """
    Solver for Robust Plane Fitting (RPF) using the
        minimum eigenvalue formulation (PCA).
    """
    def solve(
        self,
        x: np.ndarray,
    ) -> tuple[float, np.ndarray, float]:
        """
        Given a collection of points `x`, solve for the plane that
            best explains them.
        """
        start = perf_counter()

        # Solve for normal vector
        X = np.einsum("ib,jb->ij", x, x)
        e = np.linalg.eig(X)
        n = e.eigenvectors[:, 0]

        duration = perf_counter() - start

        return n, duration

### Define Convex RPF Solver

In [ ]:
class ConvexRPFSolver(Solver):
    """
    Solver for Robust Plane Fitting (RPF) using the
        semidefinite formulation.
    """
    def __init__(self, cbar2: float = 1):
        """
        Initialize this Convex RPF solver.
        """
        self.cbar2 = cbar2

    def solve(
        self,
        x: np.ndarray,
    ) -> tuple[float, np.ndarray, float]:
        """
        Given a collection of points `x`, solve for the plane that
            best explains them.
        """
        start = perf_counter()

        N = x.shape[1]
        Z = cp.Variable((3*N + 3, 3*N + 3), PSD=True)
        constraints = []

        # Write objective
        objective = 0
        for i in range(N):
            objective += cp.trace(Z[0:3, 0:3] @ np.outer(x[:, i], x[:, i]))
            objective += cp.trace(Z[0:3, i*3+3:i*3+6] @ np.outer(x[:, i], x[:, i]))
            objective += (1 - cp.trace(Z[0:3, i*3+3:i*3+6])) * self.cbar2

        # Enforce trace
        constraints.append(
            cp.trace(Z[0:3, 0:3]) == 1
        )

        # Enforce [Z]_{nn} == [Z]_{n_i n_i}
        for i in range(1, N+1):
            constraints.append(
                Z[0:3, 0:3] == Z[i*3:i*3+3, i*3:i*3+3]
            )

        # Enforce symmetry of each block
        for i in range(1, N+1):
            block = Z[0:3, i*3:i*3+3]
            constraints.append(
                block == block.T
            )
            for j in range(i, N+1):
                block = Z[j*3:j*3+3, i*3:i*3+3]
                constraints.append(
                    block == block.T
                )

        program = cp.Problem(
            cp.Minimize(objective),
            constraints,
        )
        program.solve()
        assert program.status == "optimal"
        e = np.linalg.eig(Z.value)
        rank = np.sum(np.where(np.isclose(e.eigenvalues, 0), 0, 1))
        n = e.eigenvectors[:, 0][0:3]
        n /= np.linalg.norm(n)

        duration = perf_counter() - start

        return n, duration

### Test Solver

In [ ]:
# Generate a random point cloud
pc = np.random.normal(size=(3, 100))

# Iterations per noise level
iterations = 100

# Calculate mean error over various noise levels
noise_db = np.linspace(-80, 20, 100)
errors = []

# Calculate mean runtime duration
durations = []

for n in tqdm(noise_db):
    # Calculate noise level
    signal = np.linalg.norm(np.std(pc, axis=1))
    noise_level = np.power(10, n/20) * signal
    e = []
    for _ in range(iterations):
        # Random normal
        n = np.random.normal(size=(3,))
        n /= np.linalg.norm(n)

        # Measurement noise
        noise = noise_level * np.random.normal(size=pc.shape)

        # Sampled points
        plane = pc - (n@pc)*n.reshape((3, 1))
        plane += noise

        # Solve for plane
        solver = EigenvalueRPFSolver()
        d_est, n_est, duration = solver.solve(plane)
        durations.append(duration)

        # Calculate triple geodesic error
        error = geodesic(
            n,
            n_est,
        )
        e.append(error)
    errors.append(e)

# Calculate mean error
mean_error = np.mean(np.array(errors), axis=1)

# Calculate p95 latency
p95_latency = 1e3*np.percentile(np.array(durations), 95)

# Plot mean error over noise level
fig = plt.figure()
ax = fig.add_subplot()
ax.semilogy(noise_db, mean_error)
plt.xlabel("Noise [dB]")
plt.ylabel("Double Geodesic Error")
plt.suptitle(
    f"Measurement Noise vs. Solver Error "
    f"(p95 latency: {p95_latency:.3f} ms)"
)
plt.grid(True)
plt.show()

### Run Example Convex Optimization

_Note_: running this solver with hundreds or thousands of input points may be impractical on your machine.  Future work will exploit sparsity or use operator splitting to accelerate semidefinite programming convergence.

In [ ]:
# Generate a random point cloud
pc = np.random.normal(size=(3, 12))

# Outlier probability
probability = 0.4

# Random normal
n = np.random.normal(size=(3,))
n /= np.linalg.norm(n)

# Random offset
d = np.random.normal()

# Measurement noise
noise_level = 1e-3
noise = noise_level * np.random.normal(size=pc.shape)

# Sampled points
plane = pc - (n@pc)*n.reshape((3, 1))
for i in range(plane.shape[1]):
    if np.random.random() < probability:
        plane[:, i] = 5 * np.random.normal(size=(3,))

# Solve
solver = ConvexRPFSolver(cbar2=0.1)
n_est, latency = solver.solve(plane)

print(n, n_est)